[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/magrilu/cv-dojo/blob/main/notebooks/two-views/seven-point.ipynb)

# The seven-points algorithm

The fundamental matrix has seven degrees of freedom. In the previous notebook we
paid for an eighth correspondence so that the estimation problem became _linear_.
Here we go back and ask what happens if we use exactly seven.

The answer is the reason the **seven-point algorithm** is a minimal solver:
seven correspondences leave a two-dimensional null space, hence a pencil of
matrices. The rank-two constraint then cuts that pencil with a cubic equation,
leaving one or three real fundamental matrices.


In [ ]:
#| echo: false
import sys, subprocess, itertools
from pathlib import Path

if "google.colab" in sys.modules and not Path("cv-dojo").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/magrilu/cv-dojo.git"], check=True)
    import os
    os.chdir("cv-dojo")

for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / "src" / "cvdojo").exists():
        sys.path.insert(0, str(parent / "src")); break

import numpy as np
import matplotlib.pyplot as plt

from cvdojo.house import (load_image, load_model, load_two_view_cameras,
                          load_annotation, project)
from cvdojo.plotting import clip_line_to_image, ACCENT
from cvdojo.scene import skew

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.figsize": (8, 5), "axes.grid": False})

BLUE, GREY = "#3288BD", "0.45"


# --- carried over from the residuals notebook, unchanged --------------------
def unit_F(F):
    return F / np.linalg.norm(F)


def projective_F_distance(F1, F2):
    """Chordal distance between two projective matrix classes."""
    A, B = unit_F(F1), unit_F(F2)
    return min(np.linalg.norm(A - B), np.linalg.norm(A + B))


def algebraic_residual(F, x, xp):
    return np.sum(xp * (F @ x.T).T, axis=1)


def euclidean_epipolar_residuals(F, x, xp):
    """One-sided point-to-epipolar-line residuals in views 1 and 2."""
    C = algebraic_residual(F, x, xp)
    lp = (F @ x.T).T
    l = (F.T @ xp.T).T
    n  = np.maximum(np.linalg.norm(l[:, :2],  axis=1), 1e-15)
    np_ = np.maximum(np.linalg.norm(lp[:, :2], axis=1), 1e-15)
    return np.abs(C) / n, np.abs(C) / np_


def symmetric_epipolar_residual(F, x, xp):
    rE1, rE2 = euclidean_epipolar_residuals(F, x, xp)
    return np.hypot(rE1, rE2)

model = load_model()
V3 = {k: np.array(v, float) for k, v in model["vertices"].items()}
EDGES = model["edges"]
P, Pp = load_two_view_cameras()

I  = load_image("IMG_4331.jpeg")
Ip = load_image("IMG_4337.jpeg")
H_IMG, W_IMG = I.shape[:2]

h = lambda a: np.c_[np.atleast_2d(a), np.ones(len(np.atleast_2d(a)))]

# The same hand-clicked correspondences as the eight-point notebook: real
# measurements, with whatever a mouse and a steady hand can achieve.
ann = load_annotation("two_view_matches")
xr  = h(np.array([m["x"]  for m in ann["matches"]]))
xpr = h(np.array([m["xp"] for m in ann["matches"]]))
rid = [m["id"] for m in ann["matches"]]


## Seven correspondences

Each correspondence $\{\mathbf{x}_i, \mathbf{x}_i'\}$ still gives the same linear equation

$$
\mathbf{x}_i^{'\top} \mathsf F\mathbf{x}_{i}=0.
$$

With seven correspondences the design matrix $\mathsf A$ has size $7\times9$. For points in
general position it has rank seven, so its right null space has dimension two (seven linear constraints cannot select a
single point of $\mathbb P^8$).

We work with the same noisy image correspondences used on the real images in the eight-point notebook. Among them we pick a well-conditioned subset of seven.

In [ ]:
def design_matrix(x, xp):
    """One row per correspondence: (x' tensor x)^T vec(F) = 0."""
    u,  v  = x[:, 0],  x[:, 1]
    up, vp = xp[:, 0], xp[:, 1]
    return np.column_stack([up*u, up*v, up,
                            vp*u, vp*v, vp,
                            u,    v,    np.ones(len(x))])


def normalize_points(x):
    """Centroid to the origin, mean distance to the origin equal to sqrt(2)."""
    c = x[:, :2].mean(axis=0)
    d = np.linalg.norm(x[:, :2] - c, axis=1).mean()
    s = np.sqrt(2.0) / d
    T = np.array([[s, 0, -s*c[0]],
                  [0, s, -s*c[1]],
                  [0, 0,  1.0]])
    return (T @ x.T).T, T


def subset_score(c):
    idx = list(c)
    xn, _  = normalize_points(xr[idx])
    xpn, _ = normalize_points(xpr[idx])
    s = np.linalg.svd(design_matrix(xn, xpn), compute_uv=False)
    return s[-1] / s[0]

best7 = max(itertools.combinations(range(len(rid)), 7), key=subset_score)
idx7 = list(best7)
SEVEN_IDS = [rid[i] for i in idx7]
idx_rest  = [i for i in range(len(rid)) if i not in idx7]
REST_IDS  = [rid[i] for i in idx_rest]
x7,  xp7  = xr[idx7],  xpr[idx7]
x_rest,  xp_rest  = xr[idx_rest], xpr[idx_rest]
print("seven-point sample:", " ".join(SEVEN_IDS))
print("kept aside        :", " ".join(REST_IDS))
print(f"sigma_7 / sigma_1 = {subset_score(best7):.2e}")


In [ ]:
#| echo: false
#| column: page
#| label: fig-seven-correspondences
#| fig-cap: >-
#|   The seven noisy correspondences used to run the solver, in orange,
#|   and in blue the ones kept aside.
fig, axes = plt.subplots(1, 2, figsize=(13, 7.5), layout="constrained")
for ax, im, pts, held, prime, ttl in [
        (axes[0], I,  x7,  x_rest,  False, "view 1"),
        (axes[1], Ip, xp7, xp_rest, True,  "view 2")]:
    ax.imshow(im)
    ax.scatter(pts[:, 0], pts[:, 1], s=55, facecolors="none",
               edgecolors=ACCENT, lw=2, zorder=4)
    ax.scatter(held[:, 0], held[:, 1], s=55, facecolors="none",
               edgecolors=BLUE, lw=2, zorder=4)
    for name, p, col in ([(n, q, ACCENT) for n, q in zip(SEVEN_IDS, pts)] +
                         [(n, q, BLUE)   for n, q in zip(REST_IDS, held)]):
        k = name[1:]
        lab = rf"$\mathbf{{x}}'_{{{k}}}$" if prime else rf"$\mathbf{{x}}_{{{k}}}$"
        ax.text(p[0] + 14, p[1] - 14, lab, color=col, fontsize=10)
    allp = np.vstack([pts, held])
    ax.set_xlim(allp[:, 0].min() - 160, allp[:, 0].max() + 160)
    ax.set_ylim(allp[:, 1].max() + 160, allp[:, 1].min() - 160)
    ax.set_title(ttl, fontsize=10); ax.axis("off")
plt.show()


## A pencil of fundamental matrix

Normalize the points exactly as in the eight-point algorithm and look at the
singular values of $\mathsf A$. There are seven non-zero singular values. The
 two vectors corresponding to the two zero singual values span the null space:

$$
\ker\mathsf A = \operatorname{span}(\widehat{\mathsf F}_0,
                                     \widehat{\mathsf F}_1).
$$

Every matrix

$$
\widehat{\mathsf F}(\lambda)
   = \widehat{\mathsf F}_0 + \lambda\widehat{\mathsf F}_1
$$

therefore satisfies all seven epipolar equations. Most members of this pencil,
however, have rank three and thus are not fundamental matrices.


In [ ]:
x7n, T = normalize_points(x7)
xp7n, Tp = normalize_points(xp7)
A7 = design_matrix(x7n, xp7n)
U, s7, Vt = np.linalg.svd(A7)
F0h = Vt[-2].reshape(3, 3)
F1h = Vt[-1].reshape(3, 3)

print("shape of A:", A7.shape)
print("rank of A :", np.linalg.matrix_rank(A7))
print("singular values:", np.round(s7, 5))
print("dimension of the right null space: 2")


## What's happening in $\mathbb{P}^8$

Let's consider the whole construction at once in convenient higher dimensional spaces.

A $3\times3$ matrix up to scale is a point of $\mathbb P^8$. Not every point in this space is a
fundamental matrix: it must also satisfy $\det\mathsf F = 0$, a single equation,
cubic in the entries of $\mathsf F$. The fundamental matrices therefore form a **cubic
hypersurface** in $\mathbb P^8$, of dimension seven. The cartoon illustration in  @fig-cartoon-variety attempts to illustrate our high-dimensional setup on the plane.

Each correspondence contributes one linear equation in the entries of
$\mathsf F$, that is, a **hyperplane** of $\mathbb P^8$. Seven correspondences in
general position give seven independent hyperplanes, and seven hyperplanes in
$\mathbb P^8$ meet in a line, since $8 - 7 = 1$. That line is exactly the pencil
$\widehat{\mathsf F}_0 + \lambda\widehat{\mathsf F}_1$, with $\lambda$ a
coordinate along it.

So the seven-point problem reads: **intersect a line with a cubic hypersurface**.
A line meets a hypersurface of degree $d$ in $d$ points, counted with
multiplicity and over the complex numbers. Here $d = 3$. The cubic in $\lambda$
we are about to write down is the degree of the variety we are cutting.

This also says something about the eight-points algorithm. Eight correspondences give
eight hyperplanes, which meet in a single *point* of $\mathbb P^8$, and a point
singled out by linear conditions has no reason to lie on the cubic. That is
precisely what the rank-two enforcement was doing there: the eight-point
algorithm lands just off the variety, and the truncated SVD pushes it back on.
Seven points land on a line that genuinely crosses the variety; eight points land
beside it and have to be projected.

In [ ]:
#| echo: false
#| column: page
#| label: fig-cartoon-variety
#| fig-cap: >-
#|   **Cartoon illustration**, drawn in the plane rather than in $\mathbb P^8$: a
#|   cubic curve stands in for the cubic hypersurface $\det\mathsf F = 0$, and
#|   lines stand in for hyperplanes. The counting is the same one dimension at a
#|   time: in $\mathbb P^2$ two lines meet in a point and one line meets a cubic
#|   in three points, exactly as in $\mathbb P^8$ seven hyperplanes meet in a
#|    in a line that cuts the variety three times, while eight hyperplanes meet in a point. Left: the seven-point algorithm lands on a line that
#|   crosses the variety, and every crossing is a candidate. Right: in presence of noise the
#|   eight-point algorithm  lands beside the variety and the rank-two enforcement
#|   projects it back. 
RED, GREEN = "#C0392B", "#66C2A5"
cubic = lambda x: x**3 - 3*x
xs = np.linspace(-2.12, 2.12, 700)
curve = np.stack([xs, cubic(xs)], 1)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.2), layout="constrained")

ax = axes[1]                                     # eight correspondences
ax.plot(xs, cubic(xs), color=GREEN, lw=3.2, zorder=2)
Q = np.array([0.45, 2.15])                       # where the hyperplanes meet
for m in (1.7, -1.25):          # kept away from the arrow's direction
    ax.plot(xs, m*(xs - Q[0]) + Q[1], color=BLUE, lw=1.7, alpha=.9, zorder=3)
Qp = curve[np.argmin(np.linalg.norm(curve - Q, axis=1))]
ax.annotate("", xy=Qp, xytext=Q,
            arrowprops=dict(arrowstyle="-|>", color=RED, lw=2.4, shrinkA=6, shrinkB=6))
ax.scatter(*Q, s=100, color="k", zorder=6)
ax.scatter(*Qp, s=100, color=RED, zorder=6)
ax.text(Q[0]+0.30, Q[1]-0.12, "eight hyperplanes\nmeet in a point", fontsize=11)
ax.text(Qp[0]-0.15, Qp[1]-1.30, "not on the variety:\nproject onto it",
        color=RED, fontsize=11, ha="center")
ax.set_title("eight correspondences", fontsize=13)

ax = axes[0]                                     # seven correspondences
ax.plot(xs, cubic(xs), color=GREEN, lw=3.2, zorder=2)
m, c = 0.55, 0.35
ax.plot(xs, m*xs + c, color=BLUE, lw=2.5, zorder=3)
r = np.sort([z.real for z in np.roots([1, 0, -3-m, -c]) if abs(z.imag) < 1e-9])
ax.scatter(r, m*r + c, s=115, color=ACCENT, edgecolors="k", lw=.9, zorder=6)
ax.text(-1.55, m*(-1.55)+c-1.15, "seven hyperplanes\nmeet in a line", fontsize=11)
ax.text(0.30, -2.75, "a line meets a cubic\nin three points", color=ACCENT, fontsize=11)
ax.set_title("seven correspondences", fontsize=13)

for ax in axes:
    ax.text(-2.25, 3.05, r"the variety $\det\,\mathsf{F}=0$", color=GREEN, fontsize=12)
    ax.set_xlim(-2.35, 2.35); ax.set_ylim(-3.5, 3.5)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
plt.show()

## Rank two turns into a cubic

The missing piece is the defining constraint of a fundamental matrix,

$$\det\widehat{\mathsf F}=0.$$

Substituting the pencil gives

$$
\det\bigl(\widehat{\mathsf F}_0+
           \lambda\widehat{\mathsf F}_1\bigr)=0.
$$

Every entry is linear in $\lambda$, so the determinant is a polynomial of degree
at most three. This is the only nonlinear step of the seven-point algorithm.


In [ ]:
def det_cubic(F0, F1):
    """Coefficients c0..c3 of det(F0 + lambda F1), in ascending order."""
    p = [[np.array([F0[i, j], F1[i, j]], float) for j in range(3)]
         for i in range(3)]
    mul = np.polynomial.polynomial.polymul
    add = np.polynomial.polynomial.polyadd
    sub = np.polynomial.polynomial.polysub

    m00 = sub(mul(p[1][1], p[2][2]), mul(p[1][2], p[2][1]))
    m01 = sub(mul(p[1][0], p[2][2]), mul(p[1][2], p[2][0]))
    m02 = sub(mul(p[1][0], p[2][1]), mul(p[1][1], p[2][0]))
    c = add(sub(mul(p[0][0], m00), mul(p[0][1], m01)),
            mul(p[0][2], m02))
    return np.pad(c, (0, max(0, 4-len(c))))[:4]

def pencil_roots(c, tol=1e-12):
    """Roots of c0 + c1 L + c2 L^2 + c3 L^3, with the degenerate case.

    When c3 vanishes the cubic drops degree: the missing root has run off to
    infinity, which in the pencil means F1 itself. A relative tolerance is
    essential — in floating point c3 comes out as 1e-18, never as exactly zero,
    and np.roots would happily return a root at 1e+17.
    """
    scale = np.abs(c).max()
    at_infinity = abs(c[3]) < tol * scale
    p = c[:3] if at_infinity else c
    p = np.trim_zeros(np.asarray(p, float)[::-1], "f")
    r = np.roots(p) if len(p) > 1 else np.array([])
    return r, at_infinity


coeff = det_cubic(F0h, F1h)
roots, at_infinity = pencil_roots(coeff)
real_roots = np.array([r.real for r in roots if abs(r.imag) < 1e-8])

print("det(F0 + lambda F1) coefficients [c0,c1,c2,c3]:")
print(coeff)
print(f"root at infinity (det F1 = 0)? {at_infinity}")
print("roots:")
for r in roots:
    print("  ", r)
print(f"{len(real_roots)} of them are real")


In [ ]:
#| echo: false
#| column: page
#| label: fig-seven-cubic
#| fig-cap: >-
#|   The determinant along the two-dimensional null space. The seven linear
#|   equations give the pencil; the rank-two constraint picks the intersections
#|   with det(F)=0. Depending on the data, one or three of the roots are real.
if len(real_roots):
    lo = min(real_roots.min() - 1.0, -2.0)
    hi = max(real_roots.max() + 1.0,  2.0)
else:
    lo, hi = -3, 3
lam = np.linspace(lo, hi, 600)
y = np.polynomial.polynomial.polyval(lam, coeff)
fig, ax = plt.subplots(figsize=(11.5, 4.8), layout="constrained")
ax.axhline(0, color="0.7", lw=1)
ax.plot(lam, y, color=BLUE, lw=2)
for r in real_roots:
    ax.scatter(r, 0, s=65, color=ACCENT, zorder=4)
    ax.text(r, 0, f"  {r:.2g}", va="bottom", fontsize=10)
ax.set_xlabel(r"$\lambda$")
ax.set_ylabel(r"$\det(\widehat{\mathsf{F}}_0+\lambda\widehat{\mathsf{F}}_1)$")
ax.set_title("the cubic rank constraint")
plt.show()


Each real root gives a rank-two matrix in normalized coordinates. We carry it
back to pixels with the same contravariant transformation used by the
normalized eight-point algorithm,

$$
\mathsf F = \mathsf T'^\top\widehat{\mathsf F}\mathsf T.
$$

On exact data every candidate must fit the seven chosen correspondences exactly.
Only one of them, however, is the fundamental matrix of the cameras that made the
images.

In [ ]:
def F_from_cameras(P, Pp):
    """Fundamental matrix induced by a pair of projective cameras."""
    Ch = np.linalg.svd(P)[2][-1]
    ep = Pp @ Ch
    return unit_F(skew(ep) @ Pp @ np.linalg.pinv(P))


F_true = F_from_cameras(P, Pp)

Fs, lambdas = [], []
for r in roots:
    if abs(r.imag) > 1e-8:
        continue
    Fh = F0h + r.real * F1h
    Fs.append(unit_F(Tp.T @ Fh @ T))
    lambdas.append(r.real)
if at_infinity:                      # the missing root: F1 on its own
    Fs.append(unit_F(Tp.T @ F1h @ T))
    lambdas.append(np.inf)

for j, (F, lam) in enumerate(zip(Fs, lambdas), 1):
    print(f"solution {j}: lambda={lam: .6f}   rank={np.linalg.matrix_rank(F)}   "
          f"max residual on the seven={symmetric_epipolar_residual(F, x7, xp7).max():.2e} px")

In [ ]:
#| echo: false
#| column: page
#| label: fig-seven-solutions
#| fig-cap: >-
#|   Every real root gives a valid epipolar geometry: each pencil of lines passes
#|   exactly through the seven points it was built from, and they are the same
#|   seven points in all three panels. Only the epipole moves. On these seven
#|   correspondences the three candidates are indistinguishable, which is why no
#|   quality number is shown here — we need data they have not seen.
n = len(Fs)
fig, axes = plt.subplots(1, n, figsize=(5.2*n, 6.2), layout="constrained", squeeze=False)
for j, F in enumerate(Fs):
    ax = axes[0, j]
    ax.imshow(Ip)
    for l in (F @ x7.T).T:
        seg = clip_line_to_image(l, W_IMG, H_IMG)
        if seg is not None:
            ax.plot(seg[:, 0], seg[:, 1], color=ACCENT, lw=1.1, alpha=0.85)
    ax.scatter(xp7[:, 0], xp7[:, 1], s=28, color=BLUE, zorder=3)
    ax.set_title(f"solution {j+1}      $\\lambda$ = {lambdas[j]:.4g}", fontsize=11)
    ax.axis("off")
plt.show()

## Which of the three solutions?

Every candidate reproduces the seven correspondences exactly, so the residuals on
those seven are zero for all of them and tell us nothing. We need a criterion
from somewhere else. Let us take three, in order of how much they cheat.

### Cheating: chordal distance 

We do know $\mathsf F$ here, so for a moment, we can cheat and use the chordal distance of each
candidate from it.

In [ ]:

print(f"{'':14s}{'chordal':>12s}{'lines [px]':>14s}")
for j, F in enumerate(Fs, 1):
    print(f"solution {j:<6d}{projective_F_distance(F, F_true):12.2e}")

The three chordal distances sit within a few per cent of one another,
which would suggest the candidates are nearly equivalent.

In [ ]:
#| echo: false
#| column: page
#| label: fig-candidate-entries
#| fig-cap: >-
#|   The three candidates and the true matrix, each normalised to unit Frobenius
#|   norm and sign-aligned with the true one. Colour is the order of magnitude of
#|   each entry, one shade per decade. All four show the same three tiers: a
#|   top-left block around $10^{-7}$, a last row and column around $10^{-3}$, and
#|   a bottom-right entry of one. To the eye they are the same object — and to
#|   the Frobenius norm they very nearly are.
from matplotlib.colors import BoundaryNorm

def aligned(F, ref):
    F = unit_F(F)
    return -F if np.sum(F * unit_F(ref)) < 0 else F

def fmt(v, floor):
    if abs(v) < floor:
        return f"<{floor:.0e}".replace("e-0", "e-")
    if abs(v) >= 0.1:
        return f"{v:+.2f}"
    return f"{v:.0e}".replace("e-0", "e-")

names = [f"solution {j+1}" for j in range(len(Fs))] + ["true F"]
mats  = [aligned(F, F_true) for F in Fs] + [unit_F(F_true)]

FLOOR = 1e-8                                   # below this, the entry is noise
decades = np.arange(np.log10(FLOOR), 1.0)      # one colour per order of magnitude
norm = BoundaryNorm(decades, ncolors=256)

fig, axes = plt.subplots(1, len(mats), figsize=(3.3*len(mats), 3.6),
                         layout="constrained")
for ax, nm, M in zip(axes, names, mats):
    Lm = np.log10(np.maximum(np.abs(M), FLOOR))
    im = ax.imshow(Lm, cmap="YlGnBu", norm=norm)
    for r in range(3):
        for c in range(3):
            ax.text(c, r, fmt(M[r, c], FLOOR), ha="center", va="center",
                    fontsize=8.5, color="white" if Lm[r, c] > -2.5 else "0.15")
    ax.set_xticks(range(3), [1, 2, 3], fontsize=8)
    ax.set_yticks(range(3), [1, 2, 3], fontsize=8)
    ax.set_title(nm, fontsize=11)
cb = fig.colorbar(im, ax=axes, shrink=.85, ticks=decades[::2])
cb.set_label(r"order of magnitude of $|\mathsf{F}_{ij}|$", fontsize=10)
cb.ax.set_yticklabels([f"$10^{{{int(d)}}}$" for d in decades[::2]])
plt.show()


In pixel coordinates the entries of a fundamental matrix live on very
different scales, for the same reason the design matrix did in the eight-point
notebook: the homogeneous coordinate $1$ sits next to pixel values in the
thousands, so the coefficient multiplying $u\,u'$ has to be some $f^2$ times
smaller than the one multiplying $1\cdot 1$ for the two terms to balance. Once
normalised, the top-left $2\times2$ block is of order $10^{-7}$, the last row and
column of order $10^{-3}$, and the bottom-right entry is one. That hierarchy is
the same for every fundamental matrix of this pair of images, whatever epipolar
geometry it describes — which is why, entry by entry, the four matrices look like
the same object.

The Frobenius norm knows nothing of that hierarchy: it weights every entry alike.
So the chordal distance is decided almost entirely by the last row and column,
and is essentially blind to the $2\times2$ block — which is precisely what sets
the *direction* of the epipolar lines. A small chordal distance therefore does
not certify a good epipolar geometry, and the ranking it induces is not one to
rely on.

None of this is an argument against $\mathsf F$ being a point of $\mathbb{P}^8$.
It is an argument against *measuring* there. It is better to consider distances defined in the
images.

### Cheating a little less: compare the epipolar lines

Still using the true matrix, but asking the question in the images. Both matrices
predict an epipolar line for the same point $\mathbf{x}$; how far apart are those
two lines, where the data actually is?

In [ ]:
def line_gap(F, G, x, xp):
    """How far apart, in pixels, are the epipolar lines of two matrices —
    measured at the correspondences, which is where the data is.

    For each pair, walk from the measured point x' onto the line that G predicts
    (that is the foot q), then ask how far q lies from the line that F predicts.
    One perpendicular segment per correspondence; the figure below draws it.
    """
    Lg = (G @ x.T).T
    Lg = Lg / np.linalg.norm(Lg[:, :2], axis=1, keepdims=True)
    q = xp.copy()
    q[:, :2] = xp[:, :2] - np.sum(xp * Lg, axis=1)[:, None] * Lg[:, :2]

    Lf = (F @ x.T).T
    Lf = Lf / np.linalg.norm(Lf[:, :2], axis=1, keepdims=True)
    return np.abs(np.sum(q * Lf, axis=1))

In [ ]:
#| echo: false
#| column: screen-inset
#| label: fig-line-gap
#| fig-cap: >-
#|   How two epipolar geometries are compared, for one correspondence, at the
#|   pixel scale. The white dashed line is what the true matrix predicts for
#|   $\mathbf{x}$, the orange line is what the candidate predicts, and the dot is
#|   the foot of the measured $\mathbf{x}'$ on the true line — the two are a pixel
#|   or two apart, which is the annotation error. The blue segment is what we
#|   measure: how far that foot lies from the candidate's line. All three panels
#|   are at the same scale, so the segments can be compared directly.
import matplotlib.patheffects as pe

HALO = lambda w, c="0.15": [pe.Stroke(linewidth=w, foreground=c), pe.Normal()]

gaps = [line_gap(F, F_true, xr, xpr) for F in Fs]
k_show = int(np.argmax(np.abs(gaps[0] - gaps[-1])))
xk, xpk, namek = xr[k_show], xpr[k_show], rid[k_show]

l_true = F_true @ xk
l_true = l_true / np.linalg.norm(l_true[:2])
q = xpk.copy()
q[:2] = xpk[:2] - (xpk @ l_true) * l_true[:2]

pad = max(140.0, 2.4 * max(g[k_show] for g in gaps))     # one scale for all panels

fig, axes = plt.subplots(1, len(Fs), figsize=(5.0*len(Fs), 5.0),
                         layout="constrained", squeeze=False)
for j, F in enumerate(Fs):
    ax = axes[0, j]
    lf = F @ xk
    lf = lf / np.linalg.norm(lf[:2])
    foot = q[:2] - (q @ lf) * lf[:2]

    ax.imshow(Ip, interpolation="nearest")
    for l, col, style, wid in ((l_true, "white", (0, (6, 4)), 2.6),
                               (lf,     ACCENT,  "-",         2.4)):
        seg = clip_line_to_image(l, W_IMG, H_IMG)
        if seg is not None:
            ax.plot(seg[:, 0], seg[:, 1], color=col, lw=wid, ls=style, zorder=4,
                    path_effects=HALO(wid + 2.0))
    ax.plot([q[0], foot[0]], [q[1], foot[1]], color=BLUE, lw=3.4, zorder=6,
            path_effects=HALO(5.6, "white"))
    ax.scatter(*foot, s=40, color=BLUE, edgecolors="white", lw=1.0, zorder=7)
    ax.scatter(*q[:2], s=90, color="0.25", edgecolors="white", lw=1.4, zorder=7)

    ax.set_title(f"solution {j+1}:  the two lines are {gaps[j][k_show]:.1f} px "
                 f"apart at $\\mathbf{{x}}'_{{{namek[1:]}}}$", fontsize=11)
    ax.set_xlim(q[0]-pad, q[0]+pad); ax.set_ylim(q[1]+pad, q[1]-pad)
    ax.axis("off")
plt.show()

Now the three candidates separate clearly, and in an order that the chordal
distance did not give. But we are still using $\mathsf F$, which in any real
situation we do not have.

### In practice use the correspondences kept aside (no cheating)

The correspondences kept aside played no part in the estimate. They can speak
now: the true fundamental matrix must fit them too, while a spurious root of the
cubic has no reason to. This is the criterion that survives outside a notebook,
and it is what a robust estimator does with hundreds of matches instead of three.

**Seven points create candidates; further correspondences disambiguate and
stabilise the estimate.**

In [ ]:
# One quantity throughout this section: how far each of the correspondences
# kept aside lies, in view 2, from the epipolar line a candidate predicts.
rest_res = np.array([euclidean_epipolar_residuals(F, x_rest, xp_rest)[1] for F in Fs])
rest_mean = rest_res.mean(axis=1)

hdr = "".join(f"{k:>10s}" for k in REST_IDS)
print(f"{'':12s}{hdr}{'mean':>10s}")
for j, row in enumerate(rest_res, 1):
    print(f"solution {j:<3d}" + "".join(f"{v:10.2f}" for v in row) + f"{row.mean():10.2f}")
print("\ndistances from the epipolar line in view 2, in pixels")

In [ ]:
#| echo: false
#| column: screen-inset
#| label: fig-seven-unused-point
#| fig-cap: >-
#|   What a correspondence kept aside says about each candidate. The blue circle is the point $\mathbf{x}'_i$ in view 2, and the blue segment is its distance from the
#|   epipolar line that the candidate predicts for it. The shading is that same distance evaluated at
#|   every position of view 2, with contours at 1, 2, 5, 10 and 20 pixels. Black dots are the seven
#|   seven points used for the estimate, which every candidate fits exactly.
# the one on which the candidates disagree most
k_rest = int(np.argmax(rest_res.max(axis=0) - rest_res.min(axis=0)))
x8, xp8, name8 = x_rest[k_rest], xp_rest[k_rest], REST_IDS[k_rest]

pad = 260
allp = np.vstack([xp7, xp_rest])
x0, x1 = allp[:, 0].min() - pad, allp[:, 0].max() + pad
y0, y1 = allp[:, 1].min() - pad, allp[:, 1].max() + pad

step = 4
gx, gy = np.meshgrid(np.arange(x0, x1, step), np.arange(y0, y1, step))
G = np.stack([gx, gy, np.ones_like(gx)])
LEVELS = [1, 2, 5, 10, 20]

n = len(Fs)
fig, axes = plt.subplots(1, n, figsize=(5.6*n, 5.2), layout="constrained",
                         squeeze=False)
for j, F in enumerate(Fs):
    ax = axes[0, j]
    lp = F @ x8                                   # epipolar line of x8 in view 2
    Cv = np.einsum("i,ihw->hw", lp, G)
    a2 = lp[0]**2 + lp[1]**2
    R  = np.abs(Cv) / np.sqrt(a2)        # distance from the epipolar line

    ax.imshow(Ip)
    ax.contourf(gx, gy, np.clip(R, 0, 25), levels=np.linspace(0, 25, 26),
                cmap="Spectral_r", alpha=.5)
    ax.contour(gx, gy, R, levels=LEVELS, colors="k", linewidths=.5, alpha=.6)
    foot = xp8[:2] - (xp8 @ lp) * lp[:2] / a2
    ax.plot([xp8[0], foot[0]], [xp8[1], foot[1]], color=BLUE, lw=2.8, zorder=5)
    ax.scatter(*xp8[:2], s=110, facecolors="none", edgecolors=BLUE, lw=2.6, zorder=6)
    ax.scatter(*foot, s=45, color=BLUE, zorder=6)
    ax.scatter(xp7[:, 0], xp7[:, 1], s=24, color="k", zorder=6)

    ax.set_title(f"solution {j+1}:  "
                 f"$\\mathbf{{x}}'_{{{name8[1:]}}}$ lies {rest_res[j, k_rest]:.1f} px "
                 f"from the line", fontsize=11)
    ax.set_xlim(x0, x1); ax.set_ylim(y1, y0); ax.axis("off")
plt.show()

In [ ]:
#| echo: false
#| label: fig-seven-unused-bars
#| fig-cap: >-
#|   The same distances for every correspondence kept aside, on a logarithmic
#|   scale, with
#|   the black ticks marking the mean of each candidate. One candidate sits at
#|   the level of the annotation noise on all of them; the others miss at least
#|   one badly.
w = 0.8 / len(REST_IDS)
fig, ax = plt.subplots(figsize=(8.5, 3.8), layout="constrained")
base = np.arange(len(Fs))
for t, name in enumerate(REST_IDS):
    ax.bar(base + t*w - 0.4 + w/2, rest_res[:, t], width=w, label=name)
for j, m in enumerate(rest_mean):
    ax.plot([j - 0.42, j + 0.42], [m, m], color="k", lw=1.6, zorder=5)
ax.set_yscale("log")
ax.set_xticks(base, [f"solution {i}" for i in range(1, len(Fs)+1)])
ax.set_ylabel("distance from the epipolar line [px]")
ax.legend(fontsize=8, title="kept aside", title_fontsize=8)
plt.show()

With exact data one candidate would fit the unused correspondences perfectly and
the others would not fit them at all. With clicked points the separation is not
between zero and something, but between the annotation noise and something much
larger, which is all a robust estimator ever gets to work with. Note also that a
single correspondence is not always decisive: read the whole group, not one bar.

## The algorithm

All the pieces can now be wrapped in a minimal solver. There is one important
difference from the eight-point algorithm: the output is a **list**. A minimal
solver is allowed to return several models.


In [ ]:
def seven_point(x, xp, imag_tol=1e-8, dedup_tol=1e-7):
    """Return all real fundamental matrices from exactly seven correspondences."""
    if len(x) != 7 or len(xp) != 7:
        raise ValueError("the seven-point solver needs exactly seven correspondences")

    xn, T = normalize_points(x)
    xpn, Tp = normalize_points(xp)
    _, _, Vt = np.linalg.svd(design_matrix(xn, xpn))
    F0, F1 = Vt[-2].reshape(3, 3), Vt[-1].reshape(3, 3)

    roots, at_infinity = pencil_roots(det_cubic(F0, F1))

    candidates = [F0 + r.real * F1 for r in roots if abs(r.imag) <= imag_tol]
    if at_infinity:
        candidates.append(F1)

    solutions = []
    for Fh in candidates:
        F = unit_F(Tp.T @ Fh @ T)
        if not any(projective_F_distance(F, G) < dedup_tol for G in solutions):
            solutions.append(F)
    return solutions


Fs_check = seven_point(x7, xp7)
print(f"returned {len(Fs_check)} real solution(s)")

## One root or three

The cubic has three roots over $\mathbb C$, but a real cubic can have either one
real root or three. Both cases occur, and we do not need to invent data to see
it: among all the ways of choosing seven of the clicked correspondences,
some give three real fundamental matrices and some give a single one.

When there is only one real root the minimal solver is, in a sense, lucky: the
ambiguity is there over the complex numbers, but the two spurious candidates are
not available to confuse a real algorithm.

In [ ]:
counts = {}
for c in itertools.combinations(range(len(rid)), 7):
    idx = list(c)
    xn, _  = normalize_points(xr[idx])
    xpn, _ = normalize_points(xpr[idx])
    Vt = np.linalg.svd(design_matrix(xn, xpn))[2]
    r, _ = pencil_roots(det_cubic(Vt[-2].reshape(3, 3), Vt[-1].reshape(3, 3)))
    k = sum(1 for z in r if abs(z.imag) < 1e-8)
    counts[k] = counts.get(k, 0) + 1

for k in sorted(counts):
    print(f"{counts[k]:4d} of the {sum(counts.values())} seven-point subsets "
          f"give {k} real solution(s)")

### Where the seven-point solver is used

### Where the seven-point solver is used

With noisy data, seven correspondences should not be treated as a statistically
complete estimate of $\mathsf F$. The solver is most useful as a **minimal
hypothesis generator** inside a robust estimator: sample seven matches, generate
one or three candidates, score each candidate on all the data, and keep the one
with the best support.

That is why the count of points matters so much in practice. A robust estimator
has to draw a sample in which *every* point is an inlier, and the probability of
that decays exponentially with the sample size — so seven points instead of eight
is not a curiosity, it is a smaller exponent. The same idea has produced solvers that shrink the sample
further by exploiting whatever is known in advance: a vertical direction from an
IMU, planar motion, a shared focal length, a known rotation axis.

Each of those variants is a different system of polynomial equations, and here
the subject stops being about images. Building a fast solver means eliminating
variables from a polynomial system, and a substantial line of work has grown
around doing that automatically: Gröbner-basis and action-matrix generators, the
choice of a good linear basis, syzygy-based reduction, sparse resultants,
numerical homotopy continuation, and most recently learning which homotopy path
to follow so that the spurious solutions are never computed at all. There is also
work on the prior question of *which* configurations of points and lines
constitute a minimal problem in the first place: a classification problem in
algebraic geometry.

So the seven-point algorithm is a small instance of a large family of method that are still investigated by the computer vision community.

## Further reading

- Hartley, R. and Zisserman, A. *Multiple View Geometry in Computer Vision*,
  2nd ed., Cambridge University Press, 2004. Section 11.1 for the seven- and
  eight-point algorithms.
- Hartley, R. "In defense of the eight-point algorithm", *IEEE TPAMI* 19(6),
  1997. The normalisation used here is the same one used in the previous notebook.

On minimal solvers at large:  
- Kukelova, Z., Bujnak, M. and Pajdla, T. "Automatic generator of minimal problem solvers", *ECCV*, 2008. The idea that solvers can be produced by a machine rather than by hand.
- Hrubý, P., Duff, T., Leykin, A. and Pajdla, T. "Learning to solve hard minimal problems", *CVPR*, 2022. Homotopy continuation with a learned starting point, so that the spurious solutions are never followed.
- PoseLib, <https://github.com/PoseLib/PoseLib>. A collection of the solvers in current use, worth reading as an atlas of the family.

---

**Luca Magri** — Computer Vision Dojo  
Code MIT · text and figures CC BY-NC-ND 4.0  
<https://magrilu.github.io/cv-dojo/>
